<a href="https://colab.research.google.com/github/shehan16/Machine-Learning-and-AI/blob/main/Gen%20AI/1.%20Conversational%20Data%20Analyst/Conversational_GenAI_Data_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conversational_GenAI_Data_Analyst

In [ ]:
import os
from google.colab import userdata
import glob

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
SERPER_API_KEY = userdata.get('SERPER_API_KEY')
OPENAI_API_KEY= userdata.get('OPENAI_API_KEY')

In [ ]:
# ==============================================================================
# Cell 3: Create Sample Data and Directories
# ==============================================================================
# Create necessary directories
os.makedirs("data", exist_ok=True)
os.makedirs("output/charts", exist_ok=True)

# Create sales_data.csv
sales_data_content = """OrderID,CustomerID,ProductID,Quantity,Price,Date
1,101,1,10,2.50,2023-01-15
2,102,2,5,10.00,2023-01-16
3,101,2,2,10.00,2023-01-17
4,103,3,7,5.75,2023-01-18
5,102,1,15,2.50,2023-01-19
6,104,4,3,25.00,2023-02-20
7,101,3,4,5.75,2023-02-21
8,103,2,8,10.00,2023-02-22
9,102,4,6,25.00,2023-02-23
10,104,1,12,2.50,2023-02-24
"""
with open("data/sales_data.csv", "w") as f:
    f.write(sales_data_content)

# Create customer_data.csv
customer_data_content = """CustomerID,FirstName,LastName,Email,City
101,John,Doe,john.doe@email.com,New York
102,Jane,Smith,jane.smith@email.com,Los Angeles
103,Peter,Jones,peter.jones@email.com,Chicago
104,Mary,Johnson,mary.johnson@email.com,Houston
"""
with open("data/customer_data.csv", "w") as f:
    f.write(customer_data_content)

print("Sample CSV files and directories created successfully.")

In [ ]:
# Step 1: Install necessary libraries
# In Google Colab, you might need to install openai if it's not already there.


# Step 2: Import all required libraries
import os
import pandas as pd
import openai
import re
import traceback
import json
from io import StringIO
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

# --- Configuration ---
# IMPORTANT: Set your OpenAI API Key in Colab's secrets manager.
# 1. Click the key icon (🔑) in the left sidebar.
# 2. Click "+ Add a new secret".
# 3. Name the secret "OPENAI_API_KEY".
# 4. Paste your API key into the "Value" field.
# 5. Make sure the "Notebook access" toggle is on.
try:
    from google.colab import userdata
    API_KEY = userdata.get('OPENAI_API_KEY')
except (ImportError, KeyError):
    # Fallback if not in Colab or secret not set
    API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE") # Replace with your key if not using Colab secrets

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("🚨 ERROR: OpenAI API Key not found.")
    print("Please set it in Colab's secrets manager (🔑) or replace 'YOUR_API_KEY_HERE' in the script.")
    client = None
else:
    try:
        client = openai.OpenAI(api_key=API_KEY)
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")
        client = None

# Step 3: The Agent Logic (same as before, but without the main() function)
class ConversationalDataAgent:
    """
    An agentic system to answer questions about data in multiple CSV files.
    """
    def __init__(self, file_paths: list, verbose: bool = False):
        if not client:
            raise ValueError("OpenAI client is not initialized. Please set your OPENAI_API_KEY.")
        self.file_paths = file_paths
        self.verbose = verbose
        self.chat_history = []
        self.csv_metadata = self._profiler_agent(file_paths)

    def _profiler_agent(self, file_paths: list) -> dict:
        metadata = {}
        for path in file_paths:
            try:
                df = pd.read_csv(path)
                buffer = StringIO()
                df.info(buf=buffer)
                info_str = buffer.getvalue()
                summary = (
                    f"File Path: '{path}'\n"
                    f"Columns and Data Types:\n{info_str}\n"
                    f"First 5 rows:\n{df.head().to_string()}\n"
                )
                metadata[path] = summary
            except Exception as e:
                metadata[path] = f"Error reading or profiling file: {e}"
        return metadata

    def _planner_agent(self, user_question: str) -> str:
        prompt = f"""
        You are a world-class data analyst and planner. Your job is to create a clear, step-by-step plan for a junior Python programmer to answer a user's question based on a set of CSV files.
        **User's Question:** "{user_question}"
        **Conversation History (for context):** {self.chat_history}
        **Available Data (Summaries of CSV files):**
        ---
        {json.dumps(self.csv_metadata, indent=2)}
        ---
        **Your Task:** Create a JSON object with a plan. The plan should have two keys: "thought" and "plan". The "plan" should be a clear instruction for the Coder Agent. If a chart is needed, specify the type and what the axes should represent. The final output of the code should be a single print statement.
        """
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[{"role": "system", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content

    def _coder_agent(self, plan: str) -> str:
        prompt = f"""
        You are an expert Python data scientist specializing in pandas and matplotlib. Your sole job is to write a Python script to execute a given plan.
        **The Plan:** {plan}
        **Instructions:**
        1. Write a single, complete Python script.
        2. Import all necessary libraries.
        3. If the plan requires creating a chart, save it to a file named `chart.png`.
        4. The **VERY LAST LINE** of your script **MUST** be a `print()` statement that outputs the final result (e.g., `df.to_string()`, a number, or 'chart.png').
        5. Only write the raw Python code.
        """
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[{"role": "system", "content": prompt}],
            temperature=0.0
        )
        code = response.choices[0].message.content
        match = re.search(r"```python\n(.*)\n```", code, re.DOTALL)
        if match:
            return match.group(1).strip()
        return code.strip()

    def _executor_agent(self, code: str) -> tuple[str, str]:
        old_stdout = sys.stdout
        redirected_output = sys.stdout = StringIO()
        try:
            if os.path.exists("chart.png"):
                os.remove("chart.png")
            exec(code, globals())
            sys.stdout = old_stdout
            output = redirected_output.getvalue().strip()
            return output, None
        except Exception as e:
            sys.stdout = old_stdout
            error_trace = traceback.format_exc()
            return None, error_trace

    def _summarizer_agent(self, user_question: str, code_output: str, error: str) -> str:
        if error:
            prompt = f"""
            The code execution failed. Explain the error to the user in a simple way and suggest they rephrase their question.
            **User's Question:** "{user_question}"
            **Execution Error:** {error}
            """
        else:
            prompt = f"""
            You are a helpful AI assistant. Provide a clear answer to the user's question based on the data analysis results.
            **User's Question:** "{user_question}"
            **Data Analysis Result:**
            ---
            {code_output}
            ---
            **Your Task:** Formulate a user-friendly answer. If the result is a table, present it clearly. If the result is 'chart.png', tell the user you have generated a chart for them.
            """
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "system", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content

    def ask(self, user_question: str) -> tuple[str, str]:
        try:
            plan_json_str = self._planner_agent(user_question)
            plan_data = json.loads(plan_json_str)
            plan = plan_data.get("plan", "No plan generated.")
        except Exception as e:
            return f"Sorry, I had trouble creating a plan. Error: {e}", None

        code = self._coder_agent(plan)
        result, error = self._executor_agent(code)
        final_answer = self._summarizer_agent(user_question, result, error)

        self.chat_history.append({"role": "user", "content": user_question})
        self.chat_history.append({"role": "assistant", "content": final_answer})

        # Return the final text answer and the raw result from the code executor
        return final_answer, result


# Step 4: Setup the Interactive UI for Colab
def run_agent_ui():
    if not client:
        print("Agent cannot run because the OpenAI client is not initialized.")
        return

    # --- Create Dummy CSV files for demonstration ---
    print("Creating sample CSV files: 'sales.csv' and 'products.csv'")
    sales_data = {'Date': ['2023-10-01', '2023-10-01', '2023-10-02', '2023-10-02', '2023-10-03'], 'ProductID': [101, 102, 101, 103, 102], 'Amount': [150, 200, 160, 50, 210]}
    pd.DataFrame(sales_data).to_csv('sales.csv', index=False)
    products_data = {'ProductID': [101, 102, 103, 104], 'ProductName': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'], 'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics']}
    pd.DataFrame(products_data).to_csv('products.csv', index=False)
    csv_files = ['sales.csv', 'products.csv']
    # --- End of Dummy Data Creation ---

    # Initialize the agent
    print("Initializing Agent...")
    agent = ConversationalDataAgent(file_paths=csv_files)
    print("Agent is ready. Use the chat box below.")

    # Create UI components
    chat_history_output = widgets.Output()
    prompt_input = widgets.Text(placeholder='Ask a question about the data...', layout=widgets.Layout(width='70%'))
    send_button = widgets.Button(description='Send', button_style='primary')

    def on_send_button_clicked(b):
        user_question = prompt_input.value
        if not user_question:
            return

        # Clear the input box
        prompt_input.value = ''

        # Display user's question
        with chat_history_output:
            print(f"👤 You: {user_question}")
            # Show a thinking indicator
            thinking_message = widgets.HTML("🤖 Agent is thinking...")
            display(thinking_message)

        # Get response from agent
        answer, raw_result = agent.ask(user_question)

        # Display agent's response
        with chat_history_output:
            # Remove the "thinking..." message
            clear_output(wait=True)
            # Re-display previous messages (crude way to keep history in view)
            for turn in agent.chat_history[:-2]: # all but the last exchange
                role = "👤 You" if turn['role'] == 'user' else "🤖 Agent"
                print(f"{role}: {turn['content']}")
            print(f"👤 You: {user_question}") # print the last question again
            print(f"\n🤖 Agent: {answer}")

            # If the result was a chart, display it
            if raw_result and 'chart.png' in raw_result:
                if os.path.exists('chart.png'):
                    display(Image(filename='chart.png'))

    send_button.on_click(on_send_button_clicked)

    # Arrange and display the UI
    display(widgets.VBox([
        widgets.HTML("<h2>Conversational Data Agent</h2><p>Ask questions about the sales and product data. Try 'Total sales per category' or 'Create a bar chart of total sales per product name'.</p>"),
        chat_history_output,
        widgets.HBox([prompt_input, send_button])
    ]))

# --- Run the application ---
run_agent_ui()


In [ ]:
# Step 1: Install necessary libraries
# In Google Colab, you might need to install openai if it's not already there.
# The '!' character lets you run shell commands directly from a notebook cell.
!pip install openai pandas -q

# Step 2: Import all required libraries
import os
import pandas as pd
import openai
import re
import traceback
import json
from io import StringIO
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output, Image
import glob

# --- Configuration ---
# IMPORTANT: Set your OpenAI API Key in Colab's secrets manager.
# 1. Click the key icon (🔑) in the left sidebar.
# 2. Click "+ Add a new secret".
# 3. Name the secret "OPENAI_API_KEY".
# 4. Paste your API key into the "Value" field.
# 5. Make sure the "Notebook access" toggle is on.
try:
    from google.colab import userdata
    API_KEY = userdata.get('OPENAI_API_KEY')
except (ImportError, KeyError):
    # Fallback if not in Colab or secret not set
    API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE") # Replace with your key if not using Colab secrets

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("🚨 ERROR: OpenAI API Key not found.")
    print("Please set it in Colab's secrets manager (🔑) or replace 'YOUR_API_KEY_HERE' in the script.")
    client = None
else:
    try:
        client = openai.OpenAI(api_key=API_KEY)
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")
        client = None

# Step 3: The Agent Logic (with corrected file path handling)
class ConversationalDataAgent:
    """
    An agentic system to answer questions about data in multiple CSV files.
    """
    def __init__(self, file_paths: list, verbose: bool = False):
        if not client:
            raise ValueError("OpenAI client is not initialized. Please set your OPENAI_API_KEY.")
        self.file_paths = file_paths
        self.verbose = verbose
        self.chat_history = []
        self.csv_metadata = self._profiler_agent(file_paths)

    def _profiler_agent(self, file_paths: list) -> dict:
        metadata = {}
        for path in file_paths:
            try:
                # Get the absolute path to be safe
                abs_path = os.path.abspath(path)
                df = pd.read_csv(abs_path)
                buffer = StringIO()
                df.info(buf=buffer)
                info_str = buffer.getvalue()
                summary = (
                    f"File Path: '{abs_path}'\n" # Use the absolute path in the summary
                    f"Columns and Data Types:\n{info_str}\n"
                    f"First 5 rows:\n{df.head().to_string()}\n"
                )
                metadata[abs_path] = summary # Use the absolute path as the key
            except Exception as e:
                metadata[path] = f"Error reading or profiling file: {e}"
        return metadata

    def _planner_agent(self, user_question: str) -> str:
        if self.verbose:
            print("2. Pondering the question and creating a plan (Planner Agent)...")
        prompt = f"""
        You are a world-class data analyst and planner. Your job is to create a clear, step-by-step plan for a junior Python programmer to answer a user's question based on a set of CSV files.
        **User's Question:** "{user_question}"
        **Conversation History (for context):** {self.chat_history}
        **Available Data (Summaries of CSV files with their FULL PATHS):**
        ---
        {json.dumps(self.csv_metadata, indent=2)}
        ---
        **Your Task:** Create a JSON object with a plan. The plan must have two keys: "thought" and "plan". The "plan" instruction **MUST** specify the **FULL FILE PATHS** for any files that need to be loaded. The final output of the code should be a single print statement.
        """
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[{"role": "system", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        plan_json_str = response.choices[0].message.content
        if self.verbose:
            plan_data = json.loads(plan_json_str)
            print("--- PLANNER AGENT THOUGHTS ---")
            print(plan_data.get("thought", "No thought process recorded."))
            print("------------------------------\n")
        return plan_json_str

    def _coder_agent(self, plan: str) -> str:
        if self.verbose:
            print("3. Writing Python code to execute the plan (Coder Agent)...")
        prompt = f"""
        You are an expert Python data scientist specializing in pandas and matplotlib. Your sole job is to write a Python script to execute a given plan.
        **The Plan:** {plan}
        **CRITICAL Instructions:**
        1. Write a single, complete Python script.
        2. You **MUST** use the **EXACT AND FULL FILE PATHS** provided in the plan.
        3. If the plan requires a chart, save it to `chart.png`.
        4. The **VERY LAST LINE** of your script **MUST** be a `print()` statement with the final result.
        5. Only write the raw Python code.
        """
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[{"role": "system", "content": prompt}],
            temperature=0.0
        )
        code = response.choices[0].message.content
        match = re.search(r"```python\n(.*)\n```", code, re.DOTALL)
        if match:
            return match.group(1).strip()
        return code.strip()

    def _executor_agent(self, code: str) -> tuple[str, str]:
        if self.verbose:
            print("4. Running the generated code (Executor Agent)...")
            print("--- EXECUTABLE CODE ---")
            print(code)
            print("-----------------------\n")
        old_stdout = sys.stdout
        redirected_output = sys.stdout = StringIO()
        try:
            if os.path.exists("chart.png"):
                os.remove("chart.png")
            exec(code, globals())
            sys.stdout = old_stdout
            output = redirected_output.getvalue().strip()
            if self.verbose:
                print("--- RAW EXECUTION OUTPUT ---")
                print(output)
                print("--------------------------\n")
            return output, None
        except Exception as e:
            sys.stdout = old_stdout
            error_trace = traceback.format_exc()
            if self.verbose:
                print("--- EXECUTION ERROR ---")
                print(error_trace)
                print("-----------------------\n")
            return None, error_trace

    def _summarizer_agent(self, user_question: str, code_output: str, error: str) -> str:
        if self.verbose:
            print("5. Formulating the final answer (Summarizer Agent)...")
        if error:
            prompt = f"""
            The code execution failed. Explain the error to the user in a simple way.
            **User's Question:** "{user_question}"
            **Execution Error:** {error}
            """
        else:
            prompt = f"""
            You are a helpful AI assistant. Provide a clear answer to the user's question based on the data analysis results.
            **User's Question:** "{user_question}"
            **Data Analysis Result:**
            ---
            {code_output}
            ---
            **Your Task:** Formulate a user-friendly answer. If the result is 'chart.png', tell the user you have generated a chart.
            """
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "system", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content

    def ask(self, user_question: str) -> tuple[str, str]:
        try:
            plan_json_str = self._planner_agent(user_question)
            plan_data = json.loads(plan_json_str)
            plan = plan_data.get("plan", "No plan generated.")
        except Exception as e:
            return f"Sorry, I had trouble creating a plan. Error: {e}", None

        code = self._coder_agent(plan)
        result, error = self._executor_agent(code)
        final_answer = self._summarizer_agent(user_question, result, error)

        self.chat_history.append({"role": "user", "content": user_question})
        self.chat_history.append({"role": "assistant", "content": final_answer, "raw_result": result})

        return final_answer, result


In [ ]:
# Step 4: Setup the Interactive UI for Colab
def run_agent_ui():
    if not client:
        print("Agent cannot run because the OpenAI client is not initialized.")
        return

    # --- Create Dummy CSV files inside a 'data' directory ---
    #print("Creating 'data' directory and sample CSV files...")
    data_dir = "data"
    os.makedirs(data_dir, exist_ok=True)
    # sales_data = {'Date': ['2023-10-01', '2023-10-01', '2023-10-02', '2023-10-02', '2023-10-03'], 'ProductID': [101, 102, 101, 103, 102], 'Amount': [150, 200, 160, 50, 210]}
    # pd.DataFrame(sales_data).to_csv(os.path.join(data_dir, 'sales.csv'), index=False)
    # products_data = {'ProductID': [101, 102, 103, 104], 'ProductName': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'], 'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics']}
    # pd.DataFrame(products_data).to_csv(os.path.join(data_dir, 'products.csv'), index=False)
    # search_pattern = os.path.join(data_dir, "*.csv")
    # csv_files = glob.glob(search_pattern)
    # print(f"Found CSV files: {csv_files}")
    # --- End of Dummy Data Creation ---

    folder_path = os.path.join(os.getcwd(), "data")
    search_pattern = os.path.join(folder_path, "*.csv")
    csv_files = glob.glob(search_pattern)

    # Initialize the agent
    print("\nInitializing Agent...")
    # The 'verbose' flag is now controlled by the checkbox in the UI
    agent = ConversationalDataAgent(file_paths=csv_files, verbose=True)
    print("Agent is ready. Use the chat box below.")

    # --- UI Components ---
    chat_history_output = widgets.Output()
    status_output = widgets.Output()
    prompt_input = widgets.Text(placeholder='Ask a question about the data...', layout=widgets.Layout(width='70%'))
    send_button = widgets.Button(description='Send', button_style='primary')
    show_thinking_checkbox = widgets.Checkbox(value=True, description="Show agent's thinking process")

    def on_send_button_clicked(b):
        user_question = prompt_input.value
        if not user_question:
            return

        # Update the agent's verbosity based on the checkbox
        agent.verbose = show_thinking_checkbox.value

        # Append user message to the chat history UI
        with chat_history_output:
            print(f"👤 You: {user_question}")

        # Show thinking status
        with status_output:
            clear_output(wait=True)
            print("🤖 Agent is thinking...")

        # Get response from agent
        answer, raw_result = agent.ask(user_question)

        # Clear thinking status
        with status_output:
            clear_output(wait=True)

        # Append agent's final answer to the chat history UI
        with chat_history_output:
            print(f"🤖 Agent: {answer}")
            if raw_result and 'chart.png' in raw_result:
                if os.path.exists('chart.png'):
                    display(Image(filename='chart.png'))

        # Clear the input box for the next question
        prompt_input.value = ''

    send_button.on_click(on_send_button_clicked)

    # Arrange and display the final UI
    ui_layout = widgets.VBox([
        widgets.HTML("<h2>Conversational Data Agent</h2>"),
        show_thinking_checkbox,
        chat_history_output,
        status_output,
        widgets.HBox([prompt_input, send_button])
    ])
    display(ui_layout)

# --- Run the application ---
run_agent_ui()